<a href="https://colab.research.google.com/github/VJ13SS/Agentic_AI_Code_Generator/blob/main/AgenticAICodeGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setting up the GROQ API KEY

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "Your API key"

In [ ]:
!pip install langchain langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature = 0)

Basic LLM Call

In [ ]:
response = llm.invoke("What is the capital of India?")

print(response.content)

The capital of India is **New Delhi**.


For Structured Output

In [ ]:
from pydantic import BaseModel,Field,ConfigDict

In [ ]:

#Defining the Schema
class Schema(BaseModel):
  price:float

In [ ]:
res = llm.with_structured_output(Schema).invoke("Extract the Price from the report: the price of apple is 100 Rs")

In [ ]:
print(res)

price=100.0


##Planner Node

Schema for the response

Prompts

In [ ]:
def planner_prompt(user_prompt):
  PLANNER_PROMPT = f""" Your are the planner agent. Convert the user prompt into a complete engineering project.

  User request: {user_prompt}"""
  return PLANNER_PROMPT

In [ ]:
#prompt = planner_prompt(user_prompt)

State

For maintaining states we use Field
Field is used to add extra information and validation rules to model attributes

In [ ]:
#Schema for creating a file

class File(BaseModel):
  path: str = Field(description = "The path to the file to be created or modified")
  purpose : str = Field(description = "The purpose of the file, eg: 'main application layer','data processing logic',etc")

In [ ]:

#Schema for creating the plan
class Plan(BaseModel):
  name: str = Field(description = "The name of the app to be built")
  description : str = Field(description = "A one line description of the app to be built, eg : 'A simple calculator','A Todo list app',etc")
  techstack : str = Field(description = "The tech stack to be used for the all. eg: 'python','react',etc")
  features : list[str] = Field(description = "A list of features that the app should have, e.g : 'user authentication ','data visualization ',etc")
  files : list[File] = Field(description = "A list of files to be created,each with path and purpose")

Nodes in LangGraph

1. Planner Node
2. Architect Node
3. Coder Node

In [ ]:
#creating a simple graph
from langgraph.constants import START,END
from langgraph.graph import StateGraph

In [ ]:

#Planner agent accepts a user prompt and gives plan for the prompt
def planner_agent(state):
  user_prompt = state["user_prompt"]
  resp = llm.with_structured_output(Plan).invoke(planner_prompt(user_prompt))
  if resp is None:
    raise ValueError("Architect did not return a valid response.")

  return {"plan":resp}

In [ ]:
graph = StateGraph(dict)

In [ ]:
graph.add_node("planner",planner_agent)

In [ ]:
#agent = graph.compile()

In [ ]:
#graph.set_entry_point("planner")

In [ ]:
user_prompt = "create a simple calculator web application"

In [ ]:
#result = agent.invoke({"user_prompt":user_prompt})

In [ ]:
#print(result)

##Architect Node

Gives the implementation details

In [ ]:
def architect_prompt(plan):
  ARCHITECT_PROMPT = f"""
You are the ARCHITECT agent. Given this project plan, break it down into explicit engineering tasks.

RULES:
- For each FILE in the plan, create one or more IMPLEMENTATION TASKS.
- In each task description:
    * Specify exactly what to implement.
    * Name the variables, functions, classes, and components to be defined.
    * Mention how this task depends on or will be used by previous tasks.
    * Include integration details: imports, expected function signatures, data flow.
- Order tasks so that dependencies are implemented first.
- Each step must be SELF-CONTAINED but also carry FORWARD the relevant context from earlier tasks.

Project Plan:
{plan}
    """
  return ARCHITECT_PROMPT

In [ ]:
#implementation Task

class ImplementationTask(BaseModel):
  filepath: str = Field(description = "The path to the file to be modified")
  task_description: str = Field(description = "A detailed description of the task to be performed on the file.eg:add user authentication,etc")

In [ ]:

#for the implementation steps

class TaskPlan(BaseModel):
  implementation_steps: list[ImplementationTask] = Field(description ="A list of steps to be taken to implement the task")
  model_config = ConfigDict(extra = "allow")

#ConfigDict(extra = "allow") - allows additional elements in the object of the class even though it is not allowed in the schema
#ConfigDict is used to control the behaviour
#It supports adding unknown elements

In [ ]:
def architect_agent(state):
  plan = state["plan"]
  resp = llm.with_structured_output(TaskPlan).invoke(architect_prompt(plan))
  if resp is None:
    raise ValueError("Architect did not return a valid response.")

  resp.plan = plan # i am maintaining the context
  #it is possible to add the extra node because of model.config
  return {"task_plan":resp}

In [ ]:
graph.add_node("architect", architect_agent)

In [ ]:
graph.add_edge(start_key="planner",end_key="architect")

In [ ]:
#agent = graph.compile()

In [ ]:
#result = agent.invoke({"user_prompt":user_prompt})

In [ ]:
#print(result)

##Coder Node

The node which writes the Code

In [ ]:
from typing import Optional

class CoderState(BaseModel):
  task_plan: TaskPlan = Field(description = "The plan for the task to be implemented")
  current_step_idx: int = Field(0, description = "The index of the current step in the implementation steps")
  current_file_content: Optional[str] = Field(None, description = "The content of the file currently being edited or created")

###Tools to write code onto a file

In [ ]:
import pathlib
import subprocess
from typing import Tuple

from langchain_core.tools import tool

PROJECT_ROOT = pathlib.Path.cwd() / "generated_project"


def safe_path_for_project(path: str) -> pathlib.Path:
    p = (PROJECT_ROOT / path).resolve()
    if PROJECT_ROOT.resolve() not in p.parents and PROJECT_ROOT.resolve() != p.parent and PROJECT_ROOT.resolve() != p:
        raise ValueError("Attempt to write outside project root")
    return p


@tool
def write_file(path: str, content: str) -> str:
    """Writes content to a file at the specified path within the project root."""
    p = safe_path_for_project(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        f.write(content)
    return f"WROTE:{p}"


@tool
def read_file(path: str) -> str:
    """Reads content from a file at the specified path within the project root."""
    p = safe_path_for_project(path)
    if not p.exists():
        return ""
    with open(p, "r", encoding="utf-8") as f:
        return f.read()


@tool
def get_current_directory() -> str:
    """Returns the current working directory."""
    return str(PROJECT_ROOT)


@tool
def list_files(directory: str = ".") -> str:
    """Lists all files in the specified directory within the project root."""
    p = safe_path_for_project(directory)
    if not p.is_dir():
        return f"ERROR: {p} is not a directory"
    files = [str(f.relative_to(PROJECT_ROOT)) for f in p.glob("**/*") if f.is_file()]
    return "\n".join(files) if files else "No files found."

@tool
def run_cmd(cmd: str, cwd: str = None, timeout: int = 30) -> Tuple[int, str, str]:
    """Runs a shell command in the specified directory and returns the result."""
    cwd_dir = safe_path_for_project(cwd) if cwd else PROJECT_ROOT
    res = subprocess.run(cmd, shell=True, cwd=str(cwd_dir), capture_output=True, text=True, timeout=timeout)
    return res.returncode, res.stdout, res.stderr


def init_project_root():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    return str(PROJECT_ROOT)

In [ ]:
def coder_system_prompt():
  CODER_SYSTEM_PROMPT = """
You are the CODER agent.
You are implementing a specific engineering task.
You have access to tools to read and write files.

Always:
- Review all existing files to maintain compatibility.
- Implement the FULL file content, integrating with other modules.
- Maintain consistent naming of variables, functions, and imports.
- When a module is imported from another file, ensure it exists and is implemented as described.
    """
  return CODER_SYSTEM_PROMPT

In [ ]:
from langgraph.prebuilt import create_react_agent

Create_react_agent is a helper which helps to create an agent based on the ReAct Pattern.


##ReAct means Reason + Act

Create_react_agent allows the model to think step by step,decide which too to use,execute the tool, observe the result, continue reasoning.

Here in our Code if it is a normal llm it just genrates the text but since we use the react_agent it becomes an agent which is capable of creating,reading, writing,appending to the file.

In [ ]:
coder_tools = [read_file, write_file, list_files, get_current_directory]
react_agent = create_react_agent(llm, coder_tools)
#this turns out llm into a tool using agent based on the ReAct pattern
def coder_agent(state: dict) -> dict:
    """LangGraph tool-using coder agent."""
    coder_state: CoderState = state.get("coder_state")
    if coder_state is None:
        #at first coder state will be none because the state contains only task plan
        coder_state = CoderState(task_plan=state["task_plan"], current_step_idx=0)

    steps = coder_state.task_plan.implementation_steps
    if coder_state.current_step_idx >= len(steps):
        return {"coder_state": coder_state, "status": "DONE"}

    current_task = steps[coder_state.current_step_idx]
    existing_content = read_file.run(current_task.filepath)

    system_prompt = coder_system_prompt()
    user_prompt = (
        f"Task: {current_task.task_description}\n"
        f"File: {current_task.filepath}\n"
        f"Existing content:\n{existing_content}\n"
        "Use write_file(path, content) to save your changes."
    )

    react_agent.invoke({"messages": [{"role": "system", "content": system_prompt},
                                     {"role": "user", "content": user_prompt}]})

    coder_state.current_step_idx += 1
    return {"coder_state": coder_state}

/tmp/ipykernel_31983/1975093615.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(llm, coder_tools)


In [ ]:
graph.add_node("coder",coder_agent)

In [ ]:
graph.add_edge(start_key="architect",end_key="coder")

In [ ]:

#we are adding the conditional edges thus the coder Node will continue its execution till all the implementation steps are completed
#after all the steps are completed it returns the status DONE

graph.add_conditional_edges(
    "coder",
    lambda s: "END" if s.get("status") == "DONE" else "coder",
    {"END": END, "coder": "coder"}
)

user_prompt = "create a simple calculator web application include the css styles and the javascript inside the html file itself.On each time I press the equal to button..i should get a javascript alert of the result"

In [ ]:
user_prompt = "create a simple calculator web application include the css styles and the javascript inside the html file itself.On each time I press the equal to button..i should get a javascript alert of the result"

In [ ]:
graph.set_entry_point("planner")

In [ ]:
agent = graph.compile()

In [ ]:
result = agent.invoke(input={"user_prompt":user_prompt}, config={"recursion_limit":100})

In [ ]:
#result = agent.invoke({"user_prompt":user_prompt})

In [ ]:
print(result)

{'coder_state': CoderState(task_plan=TaskPlan(implementation_steps=[ImplementationTask(filepath='index.html', task_description='Create the HTML skeleton for the calculator app.\n- Add `<!DOCTYPE html>` and `<html lang="en">` root.\n- Inside `<head>`, include `<meta charset="UTF-8">`, `<meta name="viewport" content="width=device-width, initial-scale=1.0">`, `<title>SimpleCalculator</title>`.\n- Insert an empty `<style></style>` block (will be filled in the next task) and an empty `<script></script>` block (will be filled later).\n- Inside `<body>`, add a `<div id="calculator" class="calculator">` container.\n- Within the container, create a `<div id="display" class="display"></div>` for showing the current expression.\n- Add a `<div class="buttons">` that will hold all calculator buttons (numbers 0‑9, operators +, -, *, /, a clear button `C`, and an equals button `=`). Use `<button>` elements with `data-value` attributes for numbers/operators and distinct `id`s for `clear` and `equals`.

In [ ]:
result

{'coder_state': CoderState(task_plan=TaskPlan(implementation_steps=[ImplementationTask(filepath='index.html', task_description='Create the HTML skeleton for the calculator app.\n- Add `<!DOCTYPE html>` and `<html lang="en">` root.\n- Inside `<head>`, include `<meta charset="UTF-8">`, `<meta name="viewport" content="width=device-width, initial-scale=1.0">`, `<title>SimpleCalculator</title>`.\n- Insert an empty `<style></style>` block (will be filled in the next task) and an empty `<script></script>` block (will be filled later).\n- Inside `<body>`, add a `<div id="calculator" class="calculator">` container.\n- Within the container, create a `<div id="display" class="display"></div>` for showing the current expression.\n- Add a `<div class="buttons">` that will hold all calculator buttons (numbers 0‑9, operators +, -, *, /, a clear button `C`, and an equals button `=`). Use `<button>` elements with `data-value` attributes for numbers/operators and distinct `id`s for `clear` and `equals`.